# Procesamiento de registros geomáticos con Python

**Lenguaje de Programación · Primer año de Ingeniería Geomática**

En una campaña de terreno se pueden registrar posiciones, diferencias de altura y controles de recepción en archivos CSV. En esta guía procesará esos registros para estimar un recorrido, preparar un plano, revisar circuitos de nivelación, detectar interrupciones y verificar mediciones de contornos en AutoCAD.

## Indicaciones de trabajo

- Abra y cierre los archivos con `open()` y `close()`. Use `csv` para leer y escribir los registros. Resuelva los ejercicios sin `with`, `split()` ni `strip()`.
- Puede importar `csv`, sus módulos propios y los componentes de la biblioteca estándar indicados en cada ejercicio. Use un bloque básico `try-except` donde lo indique el enunciado.
- Para cada ejercicio, cree un programa principal y al menos un módulo propio, en archivos `.py` separados. El programa principal importa y llama a las funciones del módulo. Las funciones de cálculo reciben sus datos y criterios mediante parámetros y retornan resultados que se pueden comprobar sin abrir archivos.
- Use la celda de trabajo de cada ejercicio para registrar su tabla de interfaces y una prueba breve. Indique el nombre de cada función, lo que recibe y lo que retorna. Cuando consulte documentación, anote el nombre de la función o método consultado, sus argumentos, su resultado y cómo lo aplicó.
- Los datos de práctica son simulados. Puede ejecutar la celda de la sección siguiente para crear los cinco CSV de entrada en la carpeta de trabajo del notebook. Al volver a ejecutarla, se conservan los archivos ya existentes.

## Datos de práctica

In [ ]:
datos = {
    "sendero.csv": """id_punto,este_m,norte_m
S01,720100,5850200
S02,720103,5850204
S03,720109,5850212
S04,720109,5850216
""",
    "observaciones_rtk.csv": """id_punto,este_m,norte_m,tipo_solucion,precision_m
P01,722110.23,5850305.19,fija,0.04
P02,722112.30,5850310.19,flotante,0.03
P01,722110.21,5850305.20,fija,0.02
P03,722125.00,5850315.10,fija,
P04,722130.10,5850318.05,fija,0.05
P01,722110.22,5850305.21,fija,0.02
P02,722112.29,5850310.18,fija,0.06
P05,722140.00,5850325.80,fija,-0.01
""",
    "tramos_nivelacion.csv": """circuito,distancia_m,desnivel_m
C01,90,1.243
C02,100,-0.821
C01,160,-1.239
C02,150,0.831
""",
    "recepcion_gnss.csv": """estacion,fecha_hora
GN01,2026-10-14 23:55:00
GN02,2026-10-15 00:00:00
GN02,2026-10-15 00:05:00
GN01,2026-10-15 00:10:xx
GN01,2026-10-15 00:15:00
GN02,2026-10-15 00:15:00
GN01,2026-10-15 00:20:00
""",
    "vertices_cad.csv": """recinto,orden,este_m,norte_m
Z01,1,720100,5850200
Z02,1,720120,5850200
Z01,2,720106,5850200
Z02,2,720120,5850204
Z01,3,720106,5850204
Z02,3,720122,5850204
Z01,4,720100,5850204
Z02,4,720122,5850202
Z02,5,720126,5850202
Z02,6,720126,5850200
""",
}

for nombre, contenido in datos.items():
    try:
        archivo = open(nombre, "x", encoding="utf-8")
    except FileExistsError:
        continue
    archivo.write(contenido)
    archivo.close()

## 1. Distancia recorrida en la inspección de un sendero

Una cuadrilla recorre un sendero para ubicar sectores que requieren mantención. Durante la inspección, un receptor GNSS guarda posiciones en el orden en que se recorrió el sendero. Cada posición tiene coordenadas Este y Norte en metros, que pueden tratarse como los ejes de un plano. La cuadrilla necesita estimar la distancia recorrida a partir de los segmentos rectos que unen las posiciones consecutivas.

**Entrada** — `sendero.csv`, con los campos `id_punto,este_m,norte_m`. Suponga que contiene al menos dos puntos y que las coordenadas son válidas.

Desarrolle un programa que:

1. Calcule la distancia horizontal entre cada par de puntos consecutivos.
2. Sume esas distancias para estimar la longitud del recorrido.
3. Identifique los dos puntos que delimitan el tramo más largo. En caso de empate, seleccione el primero.
4. Incorpore en el módulo propio una función que reciba las coordenadas de dos puntos y retorne su distancia. Use `math.hypot()` en esa función.

### Cómo consultar la ayuda desde el intérprete de Python

1. Abra una terminal del sistema. Ejecute `python3` en macOS o Linux, o `py` en Windows. Cuando vea el indicador `>>>`, ya está dentro del intérprete de Python.
2. Importe el módulo y pase la función a `help()`. Escriba las instrucciones que aparecen después de cada `>>>`.

   ```pycon
   >>> import math
   >>> help(math.hypot)
   ```

   Lea la firma y busque la explicación para dos valores. En `help(math.hypot)` se pasa la función a la ayuda. Si ejecuta `help(math)`, puede explorar el módulo completo. Si la ayuda se abre en un paginador, presione `q` para volver al intérprete.

3. Desde el mismo intérprete, pruebe la función con un caso conocido.

   ```pycon
   >>> math.hypot(3, 4)
   5.0
   ```

4. Si un punto tiene coordenadas `(100, 200)` y el siguiente `(103, 204)`, ¿qué dos valores debe pasar a `hypot()` para calcular la distancia entre ellos? Compruebe su respuesta y utilice esos puntos para probar la función que escriba.
5. Use `exit()` para salir del intérprete. También puede ejecutar `help()` sobre un objeto en una celda de código de Jupyter, por ejemplo `help(math.hypot)`.

Referencias de consulta — [Ayuda interactiva de Python](https://docs.python.org/es/3/library/functions.html#help) · [Documentación de math.hypot](https://docs.python.org/es/3/library/math.html#math.hypot).

### Ejercicio 1 · Tabla de interfaces y prueba

Registre qué identificó en la ayuda, los argumentos de su función de distancia y el resultado de su prueba.

## 2. Selección de observaciones para elaborar un plano

En un levantamiento para elaborar el plano de un recinto, una cuadrilla puede medir varias veces la misma esquina o marca. Por ejemplo, si la primera medición presenta una calidad insuficiente, el operador vuelve a medirla. El receptor GNSS en modo RTK, que utiliza correcciones para mejorar la posición calculada, exporta todas las observaciones. Por eso, un identificador puede aparecer en varias filas. Antes de dibujar el plano, el equipo necesita seleccionar una observación por punto.

Cada observación incluye el tipo de solución calculada por el equipo (`fija` o `flotante`) y una estimación de la precisión horizontal en metros. Para este trabajo se aceptan las soluciones `fija` con una precisión estimada mayor que cero y menor o igual a `0.05` m. Entre las observaciones aceptables de un mismo punto, se prefiere la que tiene el menor valor de precisión estimada.

**Entrada** — `observaciones_rtk.csv`, con los campos `id_punto,este_m,norte_m,tipo_solucion,precision_m`.  
**Salida** — `puntos_para_plano.csv`, con los mismos campos.

Desarrolle un programa que:

1. Clasifique como inválidos los registros cuyo valor de `precision_m` no se pueda convertir a número o sea menor o igual a cero. Capture `ValueError` y continúe con la siguiente fila. Suponga que las coordenadas y el tipo de solución son válidos.
2. Escriba como máximo una observación por punto, aplicando el criterio de selección. En caso de empate, conserve la primera observación aceptable.
3. Informe cuántas observaciones eran inválidas, cuántos puntos quedaron seleccionados y cuántos identificadores quedaron sin una observación aceptable.
4. Separe en funciones la evaluación de una observación y la selección de las mejores observaciones por punto.

### Ejercicio 2 · Tabla de interfaces y prueba

Incluya una prueba con dos observaciones aceptables de un mismo punto y un empate de precisión.

## 3. Control de cierre de circuitos de nivelación

Para comprobar las mediciones de altura en una obra, una cuadrilla realiza varios circuitos de nivelación. En cada uno comienza en una marca, pasa por otras y vuelve a la inicial. En cada tramo mide un **desnivel**, es decir, cuánto subió o bajó respecto de la marca anterior. Como el circuito termina donde empezó, la suma de los desniveles debería ser cercana a cero. Si la diferencia supera la tolerancia definida para el trabajo, el equipo revisa las mediciones antes de utilizarlas.

**Entrada** — `tramos_nivelacion.csv`, con los campos `circuito,distancia_m,desnivel_m`. Las filas de distintos circuitos pueden aparecer intercaladas. Suponga que los datos numéricos son válidos y que las distancias son positivas.  
**Salida** — `control_nivelacion.csv`, con los campos `circuito,distancia_total_m,error_cierre_mm,tolerancia_mm,resultado`.

Desarrolle un programa que:

1. Acumule la distancia y los desniveles con signo por circuito.
2. Calcule el error de cierre como el valor absoluto del desnivel acumulado, expresado en milímetros.
3. Calcule la tolerancia del ejercicio como $12\sqrt{d}$ mm, donde $d$ es la distancia total del circuito en kilómetros. Registre `conforme` cuando el error sea menor o igual a la tolerancia y `fuera de tolerancia` cuando la supere.
4. Genere una fila por circuito con `csv.DictWriter` e informe cuáles deben revisarse.
5. Importe y use `sqrt()` del módulo `math`. Consulte su ayuda en el intérprete e identifique qué valor debe recibir y en qué unidad se expresa ese valor en el ejercicio.

Referencia de consulta — [Documentación de math.sqrt](https://docs.python.org/es/3/library/math.html#math.sqrt).

### Ejercicio 3 · Tabla de interfaces y prueba

Compruebe por separado el cálculo del error de cierre y el de la tolerancia para un circuito.

## 4. Detección de periodos sin datos en estaciones GNSS

Un equipo administra estaciones GNSS que envían un registro cada cinco minutos. Si se interrumpe la transmisión, pueden faltar filas en el archivo recibido. Para decidir qué periodos revisar, el equipo compara las horas de registros consecutivos **de una misma estación**. Por ejemplo, si una estación envió registros a las `23:55` y a las `00:15` del día siguiente, entre ambos se esperaban tres registros que no llegaron.

**Entrada** — `recepcion_gnss.csv`, con los campos `estacion,fecha_hora`. Las fechas tienen un formato como `2026-10-14 23:55:00` y están expresadas en UTC. Los registros de distintas estaciones pueden aparecer intercalados. Para cada estación, las fechas válidas están en orden cronológico, sin duplicados y en instantes separados por múltiplos de cinco minutos.  
**Salida** — `intervalos_gnss.csv`, con los campos `estacion,ultimo_registro_antes,primer_registro_despues,minutos_entre_registros,registros_faltantes`.

Desarrolle un programa que:

1. Consulte la documentación de `datetime` y use `datetime.strptime()` para interpretar las fechas. Determine qué formato debe pasarle para leer el campo completo.
2. Si una fecha no se puede interpretar, capture `ValueError`, cuente la fila como inválida y continúe. Compare cada registro válido con el último registro válido de la misma estación.
3. Escriba un registro de salida cuando transcurran más de cinco minutos entre ambas observaciones. Calcule cuántos registros se esperaban **entre** esos dos instantes.
4. Informe la cantidad de filas inválidas y la estación con el mayor número de registros faltantes acumulados. Si hay empate, seleccione el código que aparezca primero en orden alfabético. Si no detecta intervalos, muestre un mensaje.
5. Pruebe la función que calcula el intervalo con dos fechas de días distintos.

Referencias de consulta — [Conversión de fechas con strptime](https://docs.python.org/es/3/library/datetime.html#datetime.datetime.strptime) · [Duración con timedelta.total_seconds](https://docs.python.org/es/3/library/datetime.html#datetime.timedelta.total_seconds).

### Ejercicio 4 · Tabla de interfaces y prueba

Registre cómo obtuvo el formato de la fecha y cómo comprobó la duración de un intervalo que cruza la medianoche.

## 5. Verificación de áreas y perímetros de un plano de AutoCAD

Para preparar el presupuesto de una plaza, una oficina técnica necesita conocer el área que pavimentará y la longitud de los bordes de cada zona. En AutoCAD, cada zona se dibujó como una **polilínea cerrada**, una figura cuyo contorno está formado, en este ejercicio, por segmentos rectos unidos. El comando `AREA` de AutoCAD permite consultar el área y el perímetro de esas polilíneas. El equipo quiere comprobar ambos valores en Python a partir de una tabla de vértices preparada desde el plano.

**Entrada** — `vertices_cad.csv`, con los campos `recinto,orden,este_m,norte_m`. Cada fila representa un vértice. Las filas de distintos recintos pueden aparecer intercaladas. El campo `orden` indica la posición del vértice dentro de su contorno.  
**Salida** — `mediciones_recintos.csv`, con los campos `recinto,cantidad_vertices,area_m2,perimetro_m`.

Suponga que las coordenadas están en metros, que cada recinto tiene al menos tres vértices distintos y que los contornos no se cruzan a sí mismos ni tienen huecos. El primer vértice no se repite al final del archivo. Los vértices pueden seguir el sentido horario o antihorario.

Desarrolle un programa que:

1. Agrupe los vértices por recinto y ordénelos por su campo `orden`, tratado como entero.
2. Calcule el **perímetro** sumando las distancias entre vértices consecutivos. Incluya el segmento que une el último vértice con el primero. Reutilice o adapte la función de distancia del ejercicio 1.
3. Calcule el **área** con la fórmula que aparece abajo. Aplique el valor absoluto a la suma final para que el resultado sea el mismo en los dos sentidos de recorrido.
4. Escriba una fila por recinto en el CSV de salida. Identifique el recinto que requiere más superficie de pavimento y el que requiere más longitud de borde. En caso de empate, use el identificador que aparezca primero en orden alfabético.
5. Separe el cálculo de área y el de perímetro en funciones que reciban los vértices ordenados. Compruebe ambas funciones con un rectángulo de 6 m por 4 m. Para contrastar el CSV de práctica, use las mediciones de control de la tabla. Si dispone del plano en AutoCAD, también puede seleccionar una polilínea cerrada en el comando `AREA` y contrastar ambos resultados.

### Cálculo del área a partir de los vértices

Para cada par de vértices consecutivos, multiplique el Este del primero por el Norte del segundo y reste el Norte del primero multiplicado por el Este del segundo. Sume las diferencias, incluyendo el par formado por el último vértice y el primero. Tome el valor absoluto de esa suma y divídalo por dos.

$$
A = \frac{1}{2}\left|\sum_{i=1}^{n}(E_iN_{i+1} - N_iE_{i+1})\right|
$$

| Recinto | Área de control (m²) | Perímetro de control (m) |
|---|---:|---:|
| Z01 | 24 | 20 |
| Z02 | 16 | 20 |

Referencia de consulta — [Comando AREA en la documentación de AutoCAD](https://help.autodesk.com/cloudhelp/2026/ENU/AutoCAD-Core/files/GUID-0591351F-8750-425C-9F1C-98B1C73D9D55.htm).

### Ejercicio 5 · Tabla de interfaces y prueba

Registre el resultado de área y perímetro del rectángulo y compruebe que ambos se mantienen al invertir el orden de sus vértices.

## Criterios de evaluación

| Aspecto | Evidencia esperada |
|---|---|
| Procesamiento de registros | El programa lee los campos correctos, aplica los criterios del problema y genera las salidas solicitadas. |
| Diseño modular | El programa principal importa el módulo propio. Cada función tiene una responsabilidad y sus parámetros y retornos permiten probarla de manera independiente. |
| Consulta de ayuda y documentación | Se identifica el objeto consultado, se explica qué recibe y qué devuelve, y se lo usa en el cálculo solicitado. |
| Tratamiento de excepciones | Se captura la excepción indicada y se aplica el comportamiento solicitado para el registro o archivo que produjo el error. |
| Verificación | La prueba incluye entradas concretas, el resultado que se esperaba y el resultado observado. |